# Streaming and Batching in LangChain (LCEL)

In LangChain, **Streaming** and **Batching** are core capabilities provided by the **Runnable** interface. These methods allow you to control how data flows through your chains—either token-by-token for responsiveness or in parallel groups for efficiency.

---

## 1. Streaming (Real-time Feedback)
Streaming allows you to surface the LLM's output as it is being generated, rather than waiting for the entire response to finish. This is crucial for user interfaces to reduce **Time to First Token (TTFT)**.

### Key Methods
* `.stream()`: Returns an iterator that yields chunks of the response.
* `.astream()`: The asynchronous version of the above.
* `.astream_events()`: An advanced method that streams intermediate steps (e.g., retrieval or tool calls) alongside the final output.

### Code Example
```python
# Standard streaming implementation
for chunk in chain.stream({"topic": "black holes"}):
    # 'end=""' ensures tokens appear side-by-side
    print(chunk.content, end="|", flush=True)
```

## 2. Batching (Parallel Processing)
Batching allows you to send multiple inputs through your chain simultaneously. LangChain optimizes this by running the inputs in parallel using a thread pool, which is significantly faster than processing items in a linear loop.

### Key Methods
* `.batch():` Takes a list of inputs and returns a list of outputs.
* `.abatch():` The asynchronous version for high-concurrency processing.
* `config:` You can limit concurrency by passing `{"max_concurrency": 5}`.

### Code Example
```python 
# Processing multiple inputs in parallel
inputs = [{"topic": "cats"}, {"topic": "dogs"}, {"topic": "birds"}]
results = chain.batch(inputs)
```

# LangChain Interface Comparison: Streaming vs. Batching

| Feature | Streaming (`.stream`) | Batching (`.batch`) |
| :--- | :--- | :--- |
| **Primary Goal** | **Responsiveness:** Reduce perceived latency by showing data immediately. | **Efficiency:** Maximize throughput by running multiple tasks in parallel. |
| **Output Behavior** | Yields data in **chunks** (tokens) as they are generated. | Returns a **list of full results** once all tasks are finished. |
| **Performance Metric** | Optimizes for **Time to First Token (TTFT)**. | Optimizes for **Total Execution Time** for multiple inputs. |
| **Common Use Cases** | Chatbots, interactive AI assistants, long-form content generation. | Document processing, data categorization, bulk summarization, evaluation. |
| **Implementation** | Requires an iterator or `async for` loop to handle partial data. | Requires a list of input dictionaries; handles threading automatically. |
| **Method Variants** | `.stream()`, `.astream()`, `.astream_events()` | `.batch()`, `.abatch()` |

### Streaming
Most models can stream their output content while it is being generated. By displaying output progressively, streaming significantly improves user experience, particularly for longer responses.
Calling stream() returns an iterator that yields output chunks as they are produced. You can use a loop to process each chunk in real-time:

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = init_chat_model("google_genai:gemini-2.5-flash")
response = model.invoke("Why do parrots talk?")
response.content

'Parrots are fascinating creatures, and their ability to mimic human speech is truly remarkable! However, it\'s important to understand that they don\'t "talk" in the same cognitive way humans do, understanding the meaning of every word. Instead, their vocalizations are a complex interplay of biological, social, and environmental factors:\n\n1.  **Unique Vocal Anatomy (Syrinx):**\n    *   Unlike humans who have a larynx (voice box), birds have a **syrinx**. Parrots, in particular, have a highly complex and muscular syrinx located deep in their chest. This allows them to produce a wide range of sounds, modulate pitch and rhythm, and mimic intricate vocal patterns.\n    *   They also have a thick, muscular tongue that helps them shape sounds, much like humans.\n\n2.  **Specialized Brain Structures for Vocal Learning:**\n    *   Parrots, along with songbirds and hummingbirds, are among the few animal groups capable of **vocal learning**. This means they can learn to produce new sounds by 

In [3]:
model.invoke("Write me a 200 words paragraph on Artificial Intelligence")

AIMessage(content="Artificial Intelligence (AI) is a transformative field dedicated to creating machines that can simulate human intelligence processes, including learning, reasoning, problem-solving, perception, and language understanding. At its core, AI enables computer systems to analyze data, identify patterns, make decisions, and adapt their behavior without explicit programming for every scenario. This broad discipline encompasses various technologies like machine learning, deep learning, natural language processing, and computer vision, each contributing to AI's remarkable capabilities.\n\nFrom powering personalized recommendations and autonomous vehicles to revolutionizing medical diagnostics and financial analysis, AI is rapidly integrating into nearly every sector. It automates complex tasks, extracts critical insights from vast datasets, and drives innovations that enhance efficiency and create new possibilities. While offering immense potential for progress and societal be

In [6]:
for chunk in model.stream("Write me a 200 words paragraph on Artificial Intelligence"):
    print(chunk.text, end="|" , flush=True)

Artificial Intelligence (AI) stands as a transformative field dedicated to equipping machines with the ability to perform tasks traditionally requiring human intellect. At its essence, AI encompasses a broad array of technologies, including machine learning, deep learning, natural language processing, and computer| vision, enabling systems to learn from data, reason, solve problems, perceive their environment, and make decisions. These capabilities allow AI to identify complex patterns, predict outcomes, understand and generate human language, and interpret visual information.

The applications| of AI are incredibly diverse and are rapidly reshaping industries worldwide. In healthcare, AI aids in diagnostics, drug discovery, and personalized medicine. Finance utilizes AI for fraud detection, risk assessment, and algorithmic trading. From powering the recommendation engines behind our favorite streaming services| and e-commerce platforms to enabling self-driving cars, optimizing supply 

In [7]:
for chunk in model.stream("Explain the langchain memory architecture in detail"):
    print(chunk.text, end="|" , flush=True)

LangChain's memory architecture is a crucial component that allows Large Language Models (LLMs) to maintain context and engage in multi-turn conversations, overcoming their inherent statelessness. It provides an abstraction layer for storing, retrieving, and managing| the history of interactions within a chain or agent.

Let's break down its architecture in detail.

---

### 1. The Core Problem: LLMs are Stateless

By design, most LLMs process one prompt at a time.| They don't inherently remember previous interactions. If you ask an LLM, "What's the capital of France?" and then immediately "And what's its population?", it wouldn't know "its" refers to France without the| previous context. Memory solves this.

### 2. The `BaseMemory` Abstraction

At the heart of LangChain's memory system is the abstract base class `BaseMemory` (located in `langchain.memory.|base`). All memory implementations inherit from this. It defines the fundamental interface for any memory system.

Key methods and 

#### Batch
Batching a collection of independent requests to a model can significantly improve performance and reduce costs, as the processing can be done in parallel:

In [9]:
response = model.batch([
    "Explain about lang chain in 200 words",
    "What is Agentic AI explain in 200 words",
    "What is Agent architecture"]
    ),
config = {'max_concurrency' : 5, # Limit to parallel calls 
          }

response

([AIMessage(content="LangChain is an open-source framework for building applications with large language models (LLMs). It addresses LLMs' inherent limitations, such as their lack of real-time data or inability to perform external actions.\n\nEssentially, LangChain acts as an orchestration layer, connecting LLMs to external data sources (databases, documents, web) for retrieval-augmented generation (RAG). This allows LLMs to answer questions using current or proprietary information, extending far beyond their initial training data.\n\nFurthermore, it integrates LLMs with external tools and APIs, enabling them to perform actions like web searches, sending emails, or interacting with other software. Agents, powered by the LLM, orchestrate these tool uses, deciding what actions to take and in what order.\n\nKey components include Chains (sequences of operations), Agents (decision-making LLMs), and various tools for data retrieval and integration. By providing these modular building blocks